# AoC 2024 Day 19 — Linen Layout

**Spark — prefix-key equi-join + a nested `aggregate` bitmask DP**

Puzzle: <https://adventofcode.com/2024/day/19>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

Two blocks. First, a comma-separated list of available towel patterns — short strings over the stripe colours `w`, `u`, `b`, `r`, `g`. Then, after a blank line, one desired design per line.

Every pattern is available in unlimited supply, and towels are laid left to right with no gaps, overlaps or reversals. So a design is **possible** exactly when it can be written as a concatenation of patterns.

- **Part 1** — count how many of the designs are possible.

In the example, 6 of the 8 designs can be built; `ubwu` and `bbrgwb` cannot.

## The approach

This is the most genuinely Spark-shaped day of the block, because the expensive part is **matching**, and matching is a join.

**Step 1 — every candidate match in one equi-join.** Explode each design into `(did, pos)` for every position, then again over `plen` from the shortest to the longest towel pattern. `substring(design, pos + 1, plen)` — with `pos` and `plen` as *columns*, not Python values — turns each `(design, position, length)` triple into a fixed string key. A single equi-join of those keys against the pattern list resolves **every towel match in the whole input at once**. No per-design loop, no UDF, one shuffle. `groupBy(did, pos)` then collects the set of lengths that fit at each position.

**Step 2 — the DP as a bitmask fold.** "Can I reach the end of this design?" is a left-to-right reachability scan, which sounds sequential. It becomes a single `aggregate` over the positions sorted ascending, carrying a **64-bit integer whose bit *i* means "position *i* is reachable"**. Seed it with bit 0. At each step: if bit `pos` is clear, the position is unreachable and the accumulator passes through untouched; if it is set, a **nested `aggregate`** over that position's matching lengths ORs in bit `pos + plen` for each. The design is possible exactly when bit `dlen` ends up set.

The nesting is the trick — `aggregate` inside `aggregate`, both native Spark expressions. The outer fold walks positions in order (that is the sequential part, and it is fine: it runs inside a single row's expression evaluation, not across a cluster), while the *rows* — the designs — stay fully parallel. Spark is doing what it is good at, one design per partition slot, and the per-design DP never leaves the JVM.

Positions absent from `steps` need no entry: nothing matches there, so they can never extend a match, and skipping them keeps the fold short.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day19

spark = get_spark('aoc-2024-day19')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = 'r, wr, b, g, bwu, rb, gb, br\n\nbrwrr\nbggr\ngbbr\nrrbgbr\nubwu\nbwurrg\nbrgr\nbbrgwb\n'

print('part 1:', day19.part1(spark, EXAMPLE), '(expected 6)')

### The join, then the fold

The first table is every towel match in the input — one row per `(design, position)` with the pattern lengths that fit there, all of it out of a single equi-join. The second shows the fold's result as raw bits: read `reachable_bits` right-to-left, and a design is possible when the bit at index `dlen` is set.

In [ ]:
from pyspark.sql import functions as F

patterns_block, designs_block = EXAMPLE.strip().split('\n\n')
designs = designs_block.splitlines()
print('patterns:', patterns_block)

designs_df, steps = day19._reachable_steps(spark, EXAMPLE)
labels = spark.createDataFrame(list(enumerate(designs)), 'did INT, design STRING')

# Every towel match in the input, resolved by one equi-join:
# (did, pos) -> the pattern lengths that fit starting there.
steps.join(labels, on='did').orderBy('did', 'pos').select(
    'did', 'design', 'pos', F.sort_array(F.col('lens')).alias('lens')
).show(24)

# The bitmask fold, exposed in binary. Bit i set = position i is reachable.
ordered = steps.groupBy('did').agg(
    F.sort_array(F.collect_list(F.struct('pos', 'lens'))).alias('steps')
)

def bit(index):
    return F.shiftleft(F.lit(1).cast('long'), index)

reached = F.aggregate(
    F.col('steps'),
    bit(F.lit(0)),
    lambda acc, step: F.when(acc.bitwiseAND(bit(step['pos'])) == 0, acc).otherwise(
        F.aggregate(step['lens'], acc, lambda a, plen: a.bitwiseOR(bit(step['pos'] + plen)))
    ),
)

ordered.join(designs_df, on='did').join(labels, on='did').select(
    'design',
    'dlen',
    F.lpad(F.bin(reached), 16, '0').alias('reachable_bits'),
    (reached.bitwiseAND(bit(F.col('dlen'))) != 0).alias('possible'),
).orderBy('did').show(truncate=False)

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 19)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day19.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day19 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- **The bitmask assumes designs are shorter than 63 characters.** The accumulator is a Spark `long`, so bit `dlen` must fit in a signed 64-bit integer. The real input tops out at 60 characters, which is why this works — but a longer design would shift past the sign bit and the design would be judged impossible with **no error raised**. This fails silently. If you point this at other input, check `max(len(d))` first.
- `substring` is **1-based**, hence `pos + 1`. The `plen` range is bounded by the shortest and longest actual pattern, so no wasted keys are generated — but note that near the end of a design `substring` returns a *short* string, which simply fails to equal any pattern of that length. That is why an equal-length key match implies the pattern genuinely fits.
- `sort_array` on an array of `struct(pos, lens)` sorts by `pos` first, which is exactly the ascending order the fold needs. That is load-bearing: fold the positions out of order and reachability propagates wrongly. It works because `pos` is the struct's first field.
- The seed is `bit(0)`, i.e. "position 0 is reachable for free" — the empty prefix. Seeding with 0 makes every design impossible.
- The `when(... == 0, acc)` guard is what makes this a correct DP rather than a greedy scan: an unreachable position must not propagate its matches forward.
- Patterns are split on `,` and stripped, so the single space after each comma in the input is handled. A pattern list with duplicates would produce duplicate join rows, but `collect_set` on `plen` collapses them.
- Part 1 only asks *whether* each design is possible, so a bitmask (one bit per position) suffices. Counting arrangements would need a per-position integer instead.